In [1]:
from google.colab import drive
drive.mount('/content/drive')  # run once per session, follow link and auth for pathes

Mounted at /content/drive


In [2]:
import json
import os
# path folders
data_path = '/content/drive/MyDrive/CrewAI_DrPatientDialogues/data_v2/'
filename = 'subjective_v2.json'
full_path = os.path.join(data_path, filename)

# Verify file exists first
if os.path.exists(full_path):
    data = json.load(open(full_path, "r"))
    print(f"Loaded {len(data)} records from {full_path}")
else:
    print(f"File not found: {full_path}")
    print("Check folder contents:")
    !ls -la "{data_path}"

Loaded 67 records from /content/drive/MyDrive/CrewAI_DrPatientDialogues/data_v2/subjective_v2.json


In [3]:
# Check first few lines
!head -5 "{full_path}"


[
  {
    "input": "Patient ID: D2N001-virtassist\nDialog: [doctor] hi , martha . how are you ?\n[patient] i'm doing okay . how are you ?\n[doctor] i'm doing okay . so , i know the nurse told you about dax . i'd like to tell dax a little bit about you , okay ?\n[patient] okay .\n[doctor] martha is a 50-year-old female with a past medical history significant for congestive heart failure , depression and hypertension who presents for her annual exam . so , martha , it's been a year since i've seen you . how are you doing ?\n[patient] i'm doing well . i've been traveling a lot recently since things have , have gotten a bit lighter . and i got my , my vaccine , so i feel safer about traveling . i've been doing a lot of hiking . uh , went to washington last weekend to hike in northern cascades, like around the mount baker area .\n[doctor] nice . that's great . i'm glad to hear that you're staying active , you know . i , i just love this weather . i'm so happy the summer is over . i'm defini

In [4]:
!pip install unsloth trl peft accelerate bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 4.0 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.1/376.1 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.7/290.7 kB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.9/122.9 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.5/170.5 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/

In [5]:
# For GPU check
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

CUDA available: True
GPU: Tesla T4


In [7]:
# fine-tuning with unsloth
from unsloth import FastLanguageModel
import torch

# A small model was used.
model_name = "unsloth/Phi-3-mini-4k-instruct-bnb-4bit"

max_seq_length = 2048  # Choose sequence length
dtype = None  # Auto detection

# Load model and tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=True,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.12.9: Fast Mistral patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.26G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/194 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/458 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [8]:
# preprocess our data: putting the data in just one single string that we can send to the model when we're fine tuning it.
from datasets import Dataset

def format_prompt(example):
    return f"### Input: {example['input']}\n### Output: {json.dumps(example['output'])}<|endoftext|>"

formatted_data = [format_prompt(item) for item in data]
dataset = Dataset.from_dict({"text": formatted_data})

In [13]:
# loading the model with add LoRA adapters:  with adding these adapters to the model we essentially going to add the kind of layers that we need, which then will actually allow us to perform the fine tuning.
model = FastLanguageModel.get_peft_model(
    model,
    r=64,  # LoRA rank - higher = more capacity, more memory
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=64 * 2,  # LoRA scaling factor (usually 2x rank)
    lora_dropout=0,  # Supports any, but = 0 is optimized
    bias="none",     # Supports any, but = "none" is optimized
    use_gradient_checkpointing="unsloth",  # Unsloth's optimized version
    random_state=3407,
    use_rslora=False,  # Rank stabilized LoRA
    loftq_config=None, # LoftQ
)

Unsloth: Already have LoRA adapters! We shall skip this step.


In [14]:
import sys
import psutil

# Find the loaded Unsloth trainer module
for name, module in sys.modules.items():
    if name.endswith("UnslothSFTTrainer"):
        module.psutil = psutil
        print("Injected psutil into:", name)

Injected psutil into: UnslothSFTTrainer


In [15]:
# seting up the supervised_fine_Tuning Trainer
from trl import SFTTrainer
from transformers import TrainingArguments

# Training arguments optimized for Unsloth
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",  # here is text because we changed our data to text
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,  # Effective batch size = 8
        warmup_steps=10,
        max_steps=60,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=25,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        save_strategy="epoch",
        save_total_limit=2,
        dataloader_pin_memory=False,
        report_to="none", # Disable Weights & Biases logging
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=4):   0%|          | 0/67 [00:00<?, ? examples/s]

In [17]:
# # --------------------------------------sample : seting up the supervised_fine_Tuning Trainer
# from trl import SFTTrainer
# from transformers import TrainingArguments

# # Training arguments optimized for Unsloth
# trainer = SFTTrainer(
#     model=model,
#     train_dataset=dataset,
#     tokenizer=tokenizer,
#     dataset_text_field="text",  # here is text because we changed our data to text
#     max_seq_length=2048,
#     args=SFTConfig(
#         per_device_train_batch_size=2,
#         gradient_accumulation_steps=4,  # Effective batch size = 8
#         warmup_steps=10,
#         max_steps=60,
#         num_train_epochs=3,
#         logging_steps=1,
#         output_dir="outputs",
#         optim="adamw_8bit"
#     )
# )

NameError: name 'SFTConfig' is not defined

In [16]:
# Train the model
trainer_stats = trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 67 | Num Epochs = 7 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 119,537,664 of 3,940,617,216 (3.03% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
25,1.662600
50,1.143500


In [ ]:
# Test the fine-tuned model
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

# Test Model 2: Medical History
messages = [
    {"role": "user", "content": """[doctor] tell me about your medical history . 
[patient] i had kidney stones about 2 years ago . same kind of pain . 
[doctor] what else ? 
[patient] i have migraines , been taking imitrex for a few months . 
[doctor] any other chronic conditions ?
[patient] yeah , acid reflux .  i'm on protonix 40mg daily .
[doctor] how are you managing the reflux ?
[patient] pretty well with diet changes , staying hydrated ."""}
]

# Expected output should focus on:
# - Past conditions (kidney stones 2 years ago)
# - Chronic medications (imitrex, protonix)
# - Management strategies

inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")

# Generate response
outputs = model.generate(input_ids=inputs, max_new_tokens=512, use_cache=True, temperature=0.7, do_sample=True, top_p=0.9)

# Decode and print
response = tokenizer.batch_decode(outputs)[0]
print(response)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


<|user|> [doctor] hi , brian . how are you ?
[patient] hi , good to see you .
[doctor] it's good to see you too . so , i know the nurse told you a little bit about dax .
[patient] mm-hmm .
[doctor] i'd like to tell dax about you , okay ?
[patient] sure .
[doctor] so , brian is a 58 year old male with a past medical history significant for congestive heart failure and hypertension , who presents today for follow-up of his chronic problems . so , brian , it's been a little while i've seen you .
[patient] mm-hmm .
[doctor] whats , what's going on ?
[patient] i , i just feel out of sorts lately . i do n't know if it's the change in the seasons or if we're just doing a lot of projects around the house and , and some , some construction on our own . i'm just feeling out of it . lack of , uh , energy . i'm just so tired and fatigued , and i feel kinda ... i feel lightheaded every once in a while .
[doctor] okay . all right . um , how long has that been going on for ?
[patient] uh , probably s

In [18]:
# Save the pre-trained model in the "gguf" format in the drive. "gguf" is the model format that ollama can undrestand(subjective_agent, history_subjective_agent, objective_agent, assessment_plan_agent)
model.save_pretrained_gguf('history_subjective_finetuned_model', tokenizer, quantization_method="q4_k_m", maximum_memory_usage=0.3)


Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

Unsloth: Preparing safetensor model files:  50%|█████     | 1/2 [03:17<03:17, 197.77s/it]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.65G [00:00<?, ?B/s]

Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [04:22<00:00, 131.19s/it]


Unsloth: Merge process complete. Saved to `/content/history_subjective_finetuned_model`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: All required system packages already installed!
Unsloth: Install llama.cpp and building - please wait 1 to 3 minutes
Unsloth: Cloning llama.cpp repository
Unsloth: Install GGUF and other packages
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['phi-3-mini-4k-in

{'save_directory': 'history_subjective_finetuned_model',
 'gguf_files': ['phi-3-mini-4k-instruct.Q4_K_M.gguf'],
 'modelfile_location': '/content/Modelfile',
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': False}

Once you confirm the `phi-3-mini-4k-instruct.Q4_K_M.gguf` file is in the specified output directory, and you have the `Modelfile` in `/content/`, you can proceed with setting up Ollama. Here are the general steps to use this model with Ollama:

1.  **Download the GGUF file:** If you are running Ollama locally, you will need to download the `phi-3-mini-4k-instruct.Q4_K_M.gguf` file from your Google Drive output directory to your local machine.

2.  **Create a Modelfile:** The `save_pretrained_gguf` function already generated a `Modelfile` in `/content/`. You will need to download this file as well. This `Modelfile` typically contains instructions for Ollama, including the `FROM` directive pointing to your GGUF file. It should look something like this:

    ```
    FROM ./phi-3-mini-4k-instruct.Q4_K_M.gguf
    # Optionally add other parameters here, e.g.:
    # PARAMETER temperature 0.7
    # PARAMETER top_k 40
    # PARAMETER top_p 0.9
    # ...
    ```

    Make sure the path in the `FROM` directive within your downloaded `Modelfile` correctly points to where you place the `phi-3-mini-4k-instruct.Q4_K_M.gguf` file on your local machine.

3.  **Create the model in Ollama:** Open your terminal (where Ollama is installed) and navigate to the directory where you saved both the `phi-3-mini-4k-instruct.Q4_K_M.gguf` file and the `Modelfile`. Then run the following command, replacing `your_model_name` with your desired name for the model in Ollama:

    ```bash
    ollama create your_model_name -f ./Modelfile
    ```

4.  **Run the model:** Once created, you can run your fine-tuned model using:

    ```bash
    ollama run your_model_name
    ```

This process tells Ollama to package your GGUF file according to the `Modelfile` instructions, making it available for use.

In [19]:
# Save the pre-trained model in the "gguf" format in the drive. "gguf" is the model format that ollama can undrestand
output_path = '/content/drive/MyDrive/CrewAI_DrPatientDialogues/output/dr_patients_q4km'
model.save_pretrained_gguf(output_path, tokenizer, quantization_method="q4_k_m")

Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files:  50%|█████     | 1/2 [04:16<04:16, 256.32s/it]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.65G [00:00<?, ?B/s]

Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [07:40<00:00, 230.38s/it]


Unsloth: Merge process complete. Saved to `/content/drive/MyDrive/CrewAI_DrPatientDialogues/output/dr_patients_q4km`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['phi-3-mini-4k-instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['phi-3-mini-4k-instruct.Q4_K_M.gguf']
Unsloth: example usage fo

{'save_directory': '/content/drive/MyDrive/CrewAI_DrPatientDialogues/output/dr_patients_q4km',
 'gguf_files': ['phi-3-mini-4k-instruct.Q4_K_M.gguf'],
 'modelfile_location': '/content/Modelfile',
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': False}

In [25]:
# model setup in ollama with adding our pre-trained model to ollama
from google.colab import files
import os

gguf_files = [f for f in os.listdir("gguf_model") if f.endswith(".gguf")]
if gguf_files:
    gguf_file = os.path.join("gguf_model", gguf_files[0])
    print(f"Downloading: {gguf_file}")
    files.download(gguf_file)

FileNotFoundError: [Errno 2] No such file or directory: 'gguf_model'

In [ ]:
# Instead of downloading directly, save the file to your Google Drive.
from google.colab import drive
drive.mount('/content/drive')

# After training, copy the file to Drive
! cp phi-3-mini-4k-instruct.Q4_K_M.gguf /content/drive/MyDrive/CrewAI_DrPatientDialogues/output/gguf
